# Backpacker AI — Basic RAG

A minimal RAG baseline for the Backpacker AI project, following the course
structure: a `search` function, a `build_prompt` function, an `llm` function,
and a top-level `rag` function that chains them together.

The knowledge base is built from [Wikivoyage](https://en.wikivoyage.org/) —
a free, community-written travel guide whose city pages are already organized
into the exact sections backpackers care about (Get in, Get around, Sleep,
Eat, Buy, Stay safe, etc.).

In [1]:
import json
import re
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

We pull plain-text extracts of ~10 popular backpacker hubs from the Wikivoyage
MediaWiki API and cache each one as JSON in `data/wikivoyage/`. Re-running the
cell is cheap — anything already on disk is skipped.

In [2]:
DESTINATIONS = [
    ("Bangkok", "Thailand"),
    ("Chiang Mai", "Thailand"),
    ("Hanoi", "Vietnam"),
    ("Ho Chi Minh City", "Vietnam"),
    ("Lisbon", "Portugal"),
    ("Barcelona", "Spain"),
    ("Mexico City", "Mexico"),
    ("Bali", "Indonesia"),
    ("Kathmandu", "Nepal"),
    ("Cusco", "Peru"),
]

DATA_DIR = Path("../data/wikivoyage")
DATA_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "BackpackerAI-Course/0.1 (course project)"


def fetch_wikivoyage_page(title):
    """Fetch a page's plain-text extract from Wikivoyage's MediaWiki API."""
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": "1",
        "exsectionformat": "wiki",
        "redirects": "1",
        "titles": title,
    }
    url = "https://en.wikivoyage.org/w/api.php?" + urlencode(params)
    req = Request(url, headers={"User-Agent": USER_AGENT})
    with urlopen(req, timeout=30) as resp:
        data = json.load(resp)
    page = next(iter(data["query"]["pages"].values()))
    return page.get("extract", "")


for city, country in DESTINATIONS:
    path = DATA_DIR / f"{city.replace(' ', '_')}.json"
    if path.exists():
        continue
    extract = fetch_wikivoyage_page(city)
    path.write_text(json.dumps(
        {"city": city, "country": country, "extract": extract},
        ensure_ascii=False, indent=2,
    ))
    print(f"Fetched {city}: {len(extract)} chars")

print(f"Cached {len(list(DATA_DIR.glob('*.json')))} city files in {DATA_DIR}")

Cached 10 city files in ../data/wikivoyage


In [3]:
SECTION_HEADER_RE = re.compile(r"^(==+)\s*(.+?)\s*\1\s*$", re.MULTILINE)

# Top-level Wikivoyage sections worth indexing for a backpacker chatbot.
RELEVANT_SECTIONS = {
    "Understand", "Get in", "Get around", "See", "Do",
    "Buy", "Eat", "Drink", "Sleep", "Stay safe", "Stay healthy",
    "Connect", "Cope", "Go next", "Talk", "Learn", "Work",
}


def split_top_sections(text):
    """Yield (section_name, body) for top-level (==) sections of a page."""
    matches = [m for m in SECTION_HEADER_RE.finditer(text) if len(m.group(1)) == 2]
    if not matches:
        return
    intro = text[:matches[0].start()].strip()
    if intro:
        yield ("Overview", intro)
    for i, m in enumerate(matches):
        name = m.group(2).strip()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[m.end():end].strip()
        if body:
            yield (name, body)


documents = []
for path in sorted(DATA_DIR.glob("*.json")):
    record = json.loads(path.read_text())
    for section, body in split_top_sections(record["extract"]):
        if section != "Overview" and section not in RELEVANT_SECTIONS:
            continue
        documents.append({
            "city": record["city"],
            "country": record["country"],
            "section": section,
            "title": f"{record['city']} — {section}",
            "content": body,
        })

print(f"Built {len(documents)} section-level documents from {len(list(DATA_DIR.glob('*.json')))} cities")
documents[0]

Built 158 section-level documents from 10 cities


{'city': 'Bali',
 'country': 'Indonesia',
 'section': 'Overview',
 'title': 'Bali — Overview',
 'content': 'Bali, the famed "Island of the Gods", is the most visited part of Indonesia. Its diverse landscape of mountainous terrain, rugged coastlines and sandy beaches, lush rice terraces and barren volcanic hillsides provide a picturesque backdrop to its colourful, spiritual and unique culture. Five rice terraces and their water temples are recognised as a  UNESCO World Heritage Site as "Cultural Landscape of Bali Province: the Subak System as a Manifestation of the Tri Hita Karana Philosophy".\nWith world-class diving and surfing, a range of natural, cultural and historical attractions, and plentiful accommodation options, it is one of the most popular island destinations in the world. Bali offers something to almost every visitor from the backpacking youth to the ultra-wealthy. Its majority-Hindu population also stands in contrast to much of the rest of majority-Muslim Indonesia.'}

## 2. Build the Index

Each document is one section of one city, so we search over the section title
and body. `city` and `country` are kept as keyword fields so we can later
filter by destination.

In [4]:
index = Index(
    text_fields=["title", "section", "content"],
    keyword_fields=["city", "country"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [5]:
def search(query):
    return index.search(query, num_results=5)

In [6]:
instructions = """
You are Backpacker AI, a no-fluff travel assistant for solo backpackers and
budget travelers. Answer the QUESTION using only the information in the
CONTEXT below — sections from Wikivoyage city pages, each tagged with its
city, country, and section (e.g. "Get in", "Sleep", "Eat", "Stay safe").

Be concise and practical. Prefer concrete numbers (prices in local currency,
journey times, distances), specific transport options (bus/train/route names),
named neighborhoods or hostel areas, visa rules, and on-the-ground tips.
Skip marketing fluff like "this charming city has something for everyone".

If the context does not contain enough to answer confidently, say so plainly
and suggest what the traveler should look up next. Always cite the city
(and section, when useful) you drew the answer from.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [7]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [8]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [9]:
print(rag("I'm in Bangkok and want to get to Chiang Mai cheaply — what are my options and roughly what should I budget?"))

To get from Bangkok to Chiang Mai cheaply, you have a couple of budget-friendly options:

### By Bus:
- **Duration**: Approx. **9 to 12 hours**, depending on the bus type.
- **Cost**: **488-550 baht** for first-class buses like Nakhonchai Air. Government buses are cheaper but less comfortable; buy tickets at **Mo Chit Bus Terminal** in Bangkok.
- **Tip**: Avoid buses advertised as "VIP" by travel agents on Khao San Road, as they might be inferior.

### By Train:
- **Duration**: **12-15 hours**; consider an overnight train to save on accommodation.
- **Cost**:
  - **Third-class**: **231 baht**
  - **Second-class**: **391 baht** (non-AC)
  - **Second-class with AC**: **641 baht**
  - **Sleeper fares** vary from **771 to 1653 baht**, depending on class and position (top/bottom bunk).
- **Tip**: Book in advance, especially for the popular overnight sleeper trains.

### Additional Tips:
- **Arrival in Chiang Mai**: You will likely arrive at **Arcade Bus Station** or **Chiang Mai Train Stati